In [ ]:
import time
import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from models.gpt2 import build_gpt2
from models.loss import gpt2_loss


def prepare_dataset(tokenizer, split="train", seq_len=256):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")[split]

    def encode(ex):
        tok = tokenizer(
            ex["text"],
            truncation=True,
            padding="max_length",
            max_length=seq_len,
        )
        return {
            "input_ids": tok["input_ids"],
            "attention_mask": tok["attention_mask"],
        }

    ds = ds.map(
        encode,
        batched=True,
        remove_columns=["text"],
        load_from_cache_file=True,
    )
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
    return ds


def train_single_gpu(num_epochs=1, bs=16, lr=2e-5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = build_gpt2(n_positions=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")  # fixed

    loader = DataLoader(
        prepare_dataset(tokenizer, "train"),
        batch_size=bs,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        prefetch_factor=None,
    )

    print(f"steps per epoch: {len(loader)}")

    total_samples = 0
    peak_mem_gb = 0.0

    for epoch in range(num_epochs):
        model.train()
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)

            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):  # fixed
                outputs = model(input_ids, attention_mask=attn, labels=input_ids)
                loss = gpt2_loss(outputs)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            window_samples += bs
            total_samples += bs

            if i % 20 == 0 and i > 0:
                if device.type == "cuda":
                    torch.cuda.synchronize()
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                if device.type == "cuda":
                    mem_gb = torch.cuda.max_memory_allocated() / 1024**3
                    peak_mem_gb = max(peak_mem_gb, mem_gb)
                    mem_str = f"mem={mem_gb:.2f}GB"
                else:
                    mem_str = ""

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss.item():.4f} "
                    f"throughput={throughput:.1f} samples/s {mem_str}"
                )

                window_start = time.perf_counter()
                window_samples = 0

        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - epoch_start
        avg_throughput = total_samples / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s | "
            f"peak_mem={peak_mem_gb:.2f}GB ===\n"
        )


if __name__ == "__main__":
    train_single_gpu(num_epochs=1, bs=16, lr=2e-5)

In [ ]:
!pwd
!ls /content
%cd /content
!git clone https://github.com/krishnajha23/training.git
%cd /content/training

/content/training
sample_data  training
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: destination path 'training' already exists and is not an empty directory.
/content/training


In [1]:
!nvidia-smi

Sat Mar 21 16:27:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.get_device_name(0))
print(f"GPU memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

Tesla T4
GPU memory total: 14.6GB


In [ ]:
import torch
import gc

# kill everything holding GPU memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"free: {torch.cuda.mem_get_info()[0]/1024**3:.2f}GB")
print(f"total: {torch.cuda.mem_get_info()[1]/1024**3:.2f}GB")

free: 14.46GB
total: 14.56GB
